# LTX Video – OpenVINO (Optimum Pipeline)

This notebook runs **LTX-Video** using `OVDiffusionPipeline` from [Optimum Intel](https://huggingface.co/docs/optimum/intel/index), matching the [official OpenVINO notebook](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/notebooks/ltx-video/ltx-video.ipynb).

**Use this to compare with the GenAI pipeline** (`run_inference.py`) if you see pixelated output.

- **Pipeline**: `OVDiffusionPipeline` (Optimum Intel)
- **Device**: CPU
- **Model path**: `LTX-Video/FP16` (reuses model from `install.sh`)

**How to run**: Open this notebook in Jupyter, use the kernel from `ltx_video_demo/.venv`, and set working dir to `ltx_video_demo`.

## 1. Install / check dependencies

Run this cell once. The venv from `install.sh` should already have most deps; this adds any missing ones for the Optimum pipeline.

In [ ]:
# Uncomment and run if packages are missing:
# %pip install -q "torch>=2.1" torchvision "transformers>=4.40" "diffusers>=0.32" "optimum-intel[openvino]" openvino huggingface-hub accelerate sentencepiece einops matplotlib "imageio[ffmpeg]"

## 2. Setup paths

In [ ]:
from pathlib import Path

# Use LTX-Video/FP16 from this folder (same as install.sh)
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "LTX-Video" / "FP16").exists():
    model_path = NOTEBOOK_DIR / "LTX-Video" / "FP16"
elif (NOTEBOOK_DIR / "ltx_video_demo" / "LTX-Video" / "FP16").exists():
    model_path = NOTEBOOK_DIR / "ltx_video_demo" / "LTX-Video" / "FP16"
else:
    raise FileNotFoundError("LTX-Video/FP16 not found. Run install.sh first.")

print(f"Model path: {model_path}")
assert (model_path / "transformer" / "openvino_model.xml").exists(), "Incomplete model. Re-run install.sh."

## 3. Load pipeline (CPU)

In [ ]:
from optimum.intel.openvino import OVDiffusionPipeline

device = "CPU"
print(f"Loading OVDiffusionPipeline on {device}...")
ov_pipe = OVDiffusionPipeline.from_pretrained(str(model_path), device=device)
print("OK: Pipeline ready.")

## 4. Generate video

Uses 704×480, 25 frames, 30 steps (matches [official notebook](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/notebooks/ltx-video/ltx-video.ipynb) and [gradio_helper](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/notebooks/ltx-video/gradio_helper.py)).

In [ ]:
import torch
from diffusers.utils import export_to_video

prompt = "A clear, turquoise river flows through a rocky canyon, cascading over a small waterfall and forming a pool of water at the bottom. The river is the main focus of the scene, with its clear water reflecting the surrounding trees and rocks. The canyon walls are steep and rocky, with some vegetation growing on them. The trees are mostly pine trees, with their green needles contrasting with the brown and gray rocks. The overall tone of the scene is one of peace and tranquility."
negative_prompt = "worst quality, inconsistent motion, blurry, jittery, distorted"
generator = torch.Generator(device="cpu").manual_seed(42)

# Same params as official OpenVINO notebook (704x480, 25 frames, 30 steps)
video = ov_pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    width=704,
    height=480,
    num_frames=25,
    num_inference_steps=30,
    generator=generator,
    guidance_scale=3,
).frames[0]

output_file = "output_ov_optimum.mp4"
export_to_video(video, output_file, fps=24)
print(f"Saved: {output_file}")

## 5. Display result

In [ ]:
from IPython.display import Video

Video("output_ov_optimum.mp4")

## Optional: Try different resolutions

If 704×480 is pixelated, try 384×384 (fewer tokens, sometimes more stable):

In [ ]:
# Uncomment and run to try 384x384:
# video = ov_pipe(
#     prompt=prompt,
#     negative_prompt=negative_prompt,
#     width=384,
#     height=384,
#     num_frames=17,
#     num_inference_steps=25,
#     generator=generator,
#     guidance_scale=3,
# ).frames[0]
# export_to_video(video, "output_ov_384.mp4", fps=24)
# Video("output_ov_384.mp4")